In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/xyz2005/qwen-data/__huggingface_repos__.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/adapter_model.safetensors
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/merges.txt
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/adapter_config.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/README.md
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/tokenizer.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/vocab.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/tokenizer_config.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/special_tokens_map.json
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/added_tokens.json
/kaggle/input/datasets/xyz2005/qwen-data/lora_output/checkpoint-867/adapter_model.safetensors
/kaggle/input/datasets/xyz2005/qwen-data/lora_output/checkpoint-867/merges.txt
/kaggle/input/datasets/xyz2005/qwen-data/lora_output/checkpoint-867/trainer_state.json
/kaggle/input/datasets/xyz2005/qwen-data/lora_output/che

In [2]:
!pip install -q transformers==4.46.3
!pip install -q peft==0.13.2
!pip install -q accelerate==1.1.1
!pip install -q sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 655.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 2.0 MB/s eta 0:00:00


In [3]:
!pip install -U bitsandbytes==0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 22.2 MB/s eta 0:00:00


In [4]:
import torch
import transformers
import peft
import accelerate
import pandas as pd

print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("Accelerate   :", accelerate.__version__)
print("Torch        :", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          torch.cuda.get_device_properties(0).total_memory / 1024**3,
          "GB")

Transformers : 4.46.3
PEFT         : 0.13.2
Accelerate   : 1.1.1
Torch        : 2.10.0+cu128

CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56219482421875 GB


In [5]:
import os
import glob
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

In [6]:
import os
import glob

adapter_files = glob.glob(
    "/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/adapter_config.json",
    recursive=True
)

print("Found adapter files:")

for f in adapter_files:
    print(f)

Found adapter files:
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora/adapter_config.json


In [7]:
assert len(adapter_files) > 0, "LoRA adapter not found!"

LORA_PATH = os.path.dirname(adapter_files[0])

print("LoRA path:")
print(LORA_PATH)

print("\nFiles:")
print(os.listdir(LORA_PATH))

LoRA path:
/kaggle/input/datasets/xyz2005/qwen-data/qwen_lora

Files:
['adapter_model.safetensors', 'merges.txt', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json']


In [8]:
test_files = glob.glob(
    "/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv",
    recursive=True
)

print("Found test files:")

for f in test_files:
    print(f)

Found test files:
/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv


In [9]:
assert len(test_files) > 0, "test_concepts.csv not found!"

TEST_CSV = test_files[0]

print("Using:")
print(TEST_CSV)

Using:
/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv


In [10]:
test_df = pd.read_csv(TEST_CSV)

print("Shape:", test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 2 rows:")
display(test_df.head(2))

Shape: (496, 11)

Columns:
['report', 'Pneumonia', 'Cardiomegaly', 'Pleural Effusion', 'Atelectasis', 'Edema', 'Pneumothorax', 'Consolidation', 'Lung Opacity', 'Nodule', 'Mass']

First 2 rows:


,report,Pneumonia,Cardiomegaly,Pleural Effusion,Atelectasis,Edema,Pneumothorax,Consolidation,Lung Opacity,Nodule,Mass
0,stable appearance of hiatal hernia. clear righ...,0,0,1,0,0,1,0,0,0,1
1,heart size normal. lungs are clear. xxxx are n...,1,0,1,0,1,1,0,0,1,1


In [11]:
LABELS = [
    "Pneumonia",
    "Cardiomegaly",
    "Pleural Effusion",
    "Atelectasis",
    "Edema",
    "Pneumothorax",
    "Consolidation",
    "Lung Opacity",
    "Nodule",
    "Mass"
]

In [12]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    LORA_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded successfully")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer loaded successfully
Pad token: <|endoftext|>
EOS token: <|endoftext|>


In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit configuration ready")

4-bit configuration ready


In [14]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Base Qwen model loaded")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base Qwen model loaded


In [15]:
model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

model.eval()

print("LoRA adapter loaded successfully!")

LoRA adapter loaded successfully!


In [16]:
model.print_trainable_parameters()

trainable params: 0 || all params: 1,562,179,072 || trainable%: 0.0000


In [17]:
def create_inference_prompt(row):

    findings = []

    for disease in LABELS:
        if int(row[disease]) == 1:
            findings.append(disease)

    if len(findings) == 0:
        findings.append("No significant abnormality")

    findings_text = "\n".join(
        [f"- {x}" for x in findings]
    )

    prompt = f"""### System
You are an expert radiologist specializing in chest X-ray interpretation.

### User
Generate a professional radiology report from the following findings.

Findings:
{findings_text}

### Assistant
"""

    return prompt

In [18]:
sample = test_df.iloc[0]

prompt = create_inference_prompt(sample)

print(prompt)

### System
You are an expert radiologist specializing in chest X-ray interpretation.

### User
Generate a professional radiology report from the following findings.

Findings:
- Pleural Effusion
- Pneumothorax
- Mass

### Assistant



In [19]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)

### System
You are an expert radiologist specializing in chest X-ray interpretation.

### User
Generate a professional radiology report from the following findings.

Findings:
- Pleural Effusion
- Pneumothorax
- Mass

### Assistant
the heart size is normal. lungs are clear bilaterally. there is no pleural line to suggest pneumothorax or costophrenic xxxx blunting to suggest large pleural effusion. there is no focal infiltrate. bony structures are within normal limits. no acute cardiopulmonary finding. . 1. no acute cardiopulmonary process. 2. no evidence for metastatic disease by known primary. if clinically indicated, further imaging of the chest may be helpful. . dr. xxxxx xxxx was called and notified of these results at xxxx p.m. xxxx xxxx. xxxx receipt of this report. dr. xxxxxxxx xxxx will notify me of any questions regarding this result. thank you for choosing xxxx as your medical specialty provider. xxxx xxxx xxxx. dr. xxxxxx xxxx xxxx at xxxx p.m. xxxx xxxx with notification of

In [20]:
def extract_report(text):

    if "### Assistant" in text:
        text = text.split("### Assistant", 1)[1]

    return text.strip()

In [21]:
generated_report = extract_report(generated_text)

print("Generated Report:\n")
print(generated_report)

Generated Report:

the heart size is normal. lungs are clear bilaterally. there is no pleural line to suggest pneumothorax or costophrenic xxxx blunting to suggest large pleural effusion. there is no focal infiltrate. bony structures are within normal limits. no acute cardiopulmonary finding. . 1. no acute cardiopulmonary process. 2. no evidence for metastatic disease by known primary. if clinically indicated, further imaging of the chest may be helpful. . dr. xxxxx xxxx was called and notified of these results at xxxx p.m. xxxx xxxx. xxxx receipt of this report. dr. xxxxxxxx xxxx will notify me of any questions regarding this result. thank you for choosing xxxx as your medical specialty provider. xxxx xxxx xxxx. dr. xxxxxx xxxx xxxx at xxxx p.m. xxxx xxxx with notification of these results. xxxx


In [22]:
print("===== GENERATED REPORT =====")
print(generated_report)

print("\n===== GROUND TRUTH REPORT =====")
print(sample["report"])

===== GENERATED REPORT =====
the heart size is normal. lungs are clear bilaterally. there is no pleural line to suggest pneumothorax or costophrenic xxxx blunting to suggest large pleural effusion. there is no focal infiltrate. bony structures are within normal limits. no acute cardiopulmonary finding. . 1. no acute cardiopulmonary process. 2. no evidence for metastatic disease by known primary. if clinically indicated, further imaging of the chest may be helpful. . dr. xxxxx xxxx was called and notified of these results at xxxx p.m. xxxx xxxx. xxxx receipt of this report. dr. xxxxxxxx xxxx will notify me of any questions regarding this result. thank you for choosing xxxx as your medical specialty provider. xxxx xxxx xxxx. dr. xxxxxx xxxx xxxx at xxxx p.m. xxxx xxxx with notification of these results. xxxx

===== GROUND TRUTH REPORT =====
stable appearance of hiatal hernia. clear right lung xxxx.in the left superior lower lobe there is a 1.9 x 1.8 cm round area of density which has inc

In [23]:
def generate_report(findings, max_new_tokens=100):

    findings_text = "\n".join(
        [f"- {finding}" for finding in findings]
    )

    prompt = f"""### System
You are an expert radiologist specializing in chest X-ray interpretation.

Generate a concise professional radiology report from the following findings.

### User
Findings:
{findings_text}

### Assistant
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,

            # Reduce repetition
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,

            # Stop generation when EOS is produced
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "### Assistant" in generated_text:
        generated_text = generated_text.split(
            "### Assistant", 1
        )[-1]

    return generated_text.strip()

In [24]:
results = []

for i in range(min(10, len(test_df))):

    print(f"Generating report {i+1}/10...")

    generated = generate_report(test_df.iloc[i])

    results.append(generated)

print("\nGeneration completed!")

Generating report 1/10...
Generating report 2/10...
Generating report 3/10...
Generating report 4/10...
Generating report 5/10...
Generating report 6/10...
Generating report 7/10...
Generating report 8/10...
Generating report 9/10...
Generating report 10/10...

Generation completed!


In [25]:
for i in range(len(results)):

    print("=" * 80)

    print(f"CASE {i+1}")

    print("\nGENERATED:")
    print(results[i])

    print("\nGROUND TRUTH:")
    print(test_df.iloc[i]["report"])

    print()

CASE 1

GENERATED:
the heart size remains within normal limits. the lungs remain grossly clear without evidence for acute infiltrate or significant change since comparison study was performed on xxxx. however, there appears to be some increase in prominence of the mediastinal contours bilaterally suggesting mild vascular congestion versus perihilar opacities consistent with emphysema. additionally, there may have been slight improvement in visualized hiatal hiatus hernia noted previously. otherwise, no definite metastatic disease identified. if clinically indicated consider

GROUND TRUTH:
stable appearance of hiatal hernia. clear right lung xxxx.in the left superior lower lobe there is a 1.9 x 1.8 cm round area of density which has increased in size compared to prior chest radiograph and recommend a xxxx chest, abdomen and pelvis with contrast as this area is suspicious for potential malignancy. normal cardiac contour. no pneumothorax or pleural effusion. 1. round area of density measu

In [26]:
generated_reports = []

for i in range(len(test_df)):

    if i % 25 == 0:
        print(f"Processing {i}/{len(test_df)}")

    report = generate_report(test_df.iloc[i])

    generated_reports.append(report)

print("All reports generated!")

Processing 0/496
Processing 25/496
Processing 50/496
Processing 75/496
Processing 100/496
Processing 125/496
Processing 150/496
Processing 175/496
Processing 200/496
Processing 225/496
Processing 250/496
Processing 275/496
Processing 300/496
Processing 325/496
Processing 350/496
Processing 375/496
Processing 400/496
Processing 425/496
Processing 450/496
Processing 475/496
All reports generated!


In [27]:
results_df = test_df.copy()

results_df["generated_report"] = generated_reports

print(results_df.shape)

display(
    results_df[
        ["report", "generated_report"]
    ].head()
)

(496, 12)


,report,generated_report
0,stable appearance of hiatal hernia. clear righ...,the heart size remains within normal limits. t...
1,heart size normal. lungs are clear. xxxx are n...,the cardiac silhouette is near top normal in s...
2,the heart size and pulmonary vascularity appea...,the cardiac silhouette measures upper limit of...
3,cardiac and mediastinal contours are within no...,the heart is topologically xxxx but otherwise ...
4,"heart xxxx, mediastinum, xxxx, bony structures...",the lungs appear clear. there is mild hyperexp...


In [28]:
OUTPUT_PATH = "/kaggle/working/generated_reports.csv"

results_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:")
print(OUTPUT_PATH)

Saved:
/kaggle/working/generated_reports.csv


In [29]:
evaluation_df = results_df[
    ["report", "generated_report"]
].copy()

evaluation_df.to_csv(
    "/kaggle/working/report_generation_results.csv",
    index=False
)

print("Evaluation file saved!")

Evaluation file saved!


In [30]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'generated_reports.csv', 'report_generation_results.csv']


In [31]:
import shutil

shutil.make_archive(
    "/kaggle/working/notebook7_results",
    "zip",
    "/kaggle/working",
    base_dir="."
)

print("ZIP created:")
print("/kaggle/working/notebook7_results.zip")

ZIP created:
/kaggle/working/notebook7_results.zip
